# Бизнес-кейс №3

Необходимо предсказать, согласится ли клиент открыть срочный вклад по итогам телемаркетинговой кампании, используя демографические признаки, историю контактов и экономические индикаторы

***Задача бинарной классификации***

## Цели и задачи:
1. Оценить качество данных:
    - пропуски данных
    - дубликаты
    - выбросы
2. Очистка/исправление аномалий
3. Ананлиз распределения признаков
4. Исследование взаимосвязи между признаками (корреляция)
5. Сегментировать данные по регионам/экономическо и сравнивать профили с помощью radar chart.
6. Формулировать бизнес-инсайты на основе данных («Что делает песню хитом?»).

## Импорты и настройки

In [33]:
# Основные библиотеки
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from math import pi

# Загрузка данных
from ucimlrepo import fetch_ucirepo 

## Превью DF

In [55]:
# fetch dataset 
bank_marketing = fetch_ucirepo(id=222) 
  
# data (as pandas dataframes) 
X = bank_marketing.data.features 
y = bank_marketing.data.targets 
  
df = pd.concat([X, y], axis=1)
df.head()

,age,job,marital,education,default,balance,housing,loan,contact,day_of_week,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,NaN,5,may,261,1,-1,0,NaN,no
1,44,technician,single,secondary,no,29,yes,no,NaN,5,may,151,1,-1,0,NaN,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,NaN,5,may,76,1,-1,0,NaN,no
3,47,blue-collar,married,NaN,no,1506,yes,no,NaN,5,may,92,1,-1,0,NaN,no
4,33,NaN,single,NaN,no,1,no,no,NaN,5,may,198,1,-1,0,NaN,no


### Описание столбцов датасета

In [56]:
pd.set_option('display.max_colwidth', None)
print(list(df.columns))
display(bank_marketing.variables)

['age', 'job', 'marital', 'education', 'default', 'balance', 'housing', 'loan', 'contact', 'day_of_week', 'month', 'duration', 'campaign', 'pdays', 'previous', 'poutcome', 'y']


,name,role,type,demographic,description,units,missing_values
0,age,Feature,Integer,Age,None,None,no
1,job,Feature,Categorical,Occupation,"type of job (categorical: 'admin.','blue-collar','entrepreneur','housemaid','management','retired','self-employed','services','student','technician','unemployed','unknown')",None,no
2,marital,Feature,Categorical,Marital Status,"marital status (categorical: 'divorced','married','single','unknown'; note: 'divorced' means divorced or widowed)",None,no
3,education,Feature,Categorical,Education Level,"(categorical: 'basic.4y','basic.6y','basic.9y','high.school','illiterate','professional.course','university.degree','unknown')",None,no
4,default,Feature,Binary,None,has credit in default?,None,no
5,balance,Feature,Integer,None,average yearly balance,euros,no
6,housing,Feature,Binary,None,has housing loan?,None,no
7,loan,Feature,Binary,None,has personal loan?,None,no
8,contact,Feature,Categorical,None,"contact communication type (categorical: 'cellular','telephone')",None,yes
9,day_of_week,Feature,Date,None,last contact day of the week,None,no


### Выводы
Признаки в наборе делятся на три логические категории:
- **Данные о клиенте.** Возраст (age), род занятий (job), семейное положение (marital), образование (education), наличие дефолта (default), среднегодовой баланс в евро (balance), наличие ипотеки (housing) и потребительского кредита (loan).

- **Детали последнего контакта.** Тип связи (contact — мобильный или городской), день месяца (day), месяц (month) и длительность звонка в секундах (duration).

- **История маркетинговых кампаний.** Количество контактов с клиентом в текущей кампании (campaign), число дней с прошлого звонка (pdays), количество прошлых контактов (previous) и результат предыдущей кампании (poutcome).

## Первичный осмотр данных 

### Первый взгляд на данные

In [57]:
df.head(10)

,age,job,marital,education,default,balance,housing,loan,contact,day_of_week,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,NaN,5,may,261,1,-1,0,NaN,no
1,44,technician,single,secondary,no,29,yes,no,NaN,5,may,151,1,-1,0,NaN,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,NaN,5,may,76,1,-1,0,NaN,no
3,47,blue-collar,married,NaN,no,1506,yes,no,NaN,5,may,92,1,-1,0,NaN,no
4,33,NaN,single,NaN,no,1,no,no,NaN,5,may,198,1,-1,0,NaN,no
5,35,management,married,tertiary,no,231,yes,no,NaN,5,may,139,1,-1,0,NaN,no
6,28,management,single,tertiary,no,447,yes,yes,NaN,5,may,217,1,-1,0,NaN,no
7,42,entrepreneur,divorced,tertiary,yes,2,yes,no,NaN,5,may,380,1,-1,0,NaN,no
8,58,retired,married,primary,no,121,yes,no,NaN,5,may,50,1,-1,0,NaN,no
9,43,technician,single,secondary,no,593,yes,no,NaN,5,may,55,1,-1,0,NaN,no


In [58]:
df.sample(30, random_state=50)

,age,job,marital,education,default,balance,housing,loan,contact,day_of_week,month,duration,campaign,pdays,previous,poutcome,y
33955,32,services,married,secondary,no,386,no,no,cellular,30,apr,222,1,-1,0,NaN,no
34904,26,management,single,tertiary,no,1494,yes,no,cellular,6,may,477,2,-1,0,NaN,no
5547,38,blue-collar,married,secondary,no,1102,yes,no,NaN,23,may,159,3,-1,0,NaN,no
15633,41,technician,married,secondary,no,525,yes,no,cellular,21,jul,108,1,-1,0,NaN,no
41402,19,student,single,secondary,no,4,no,no,cellular,3,sep,114,1,-1,0,NaN,no
20608,43,management,single,tertiary,no,1627,no,no,cellular,12,aug,105,4,-1,0,NaN,no
38208,24,admin.,married,secondary,no,-94,yes,no,cellular,15,may,217,1,-1,0,NaN,no
30621,36,unemployed,married,primary,no,0,no,no,cellular,6,feb,409,1,-1,0,NaN,no
8247,44,blue-collar,divorced,primary,yes,-119,yes,no,NaN,2,jun,132,5,-1,0,NaN,no
10630,50,blue-collar,married,secondary,no,887,no,no,NaN,16,jun,287,1,-1,0,NaN,no


#### Что бросатеся в глаза в результате беглого осмотра срезов и требует исследования:
1. pdays = -1
2. poutcome = NaN
3. previous = 0


### Технический паспорт данных

In [59]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45211 entries, 0 to 45210
Data columns (total 17 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   age          45211 non-null  int64 
 1   job          44923 non-null  object
 2   marital      45211 non-null  object
 3   education    43354 non-null  object
 4   default      45211 non-null  object
 5   balance      45211 non-null  int64 
 6   housing      45211 non-null  object
 7   loan         45211 non-null  object
 8   contact      32191 non-null  object
 9   day_of_week  45211 non-null  int64 
 10  month        45211 non-null  object
 11  duration     45211 non-null  int64 
 12  campaign     45211 non-null  int64 
 13  pdays        45211 non-null  int64 
 14  previous     45211 non-null  int64 
 15  poutcome     8252 non-null   object
 16  y            45211 non-null  object
dtypes: int64(7), object(10)
memory usage: 5.9+ MB


### Анализ `.info()`:

| Метрика | Значение | Вывод |
|---------|----------|-------|
| **Типы данных** | int64(7), object(10) | OK, но можно оптимизировать |
| **Пропуски** | Есть в `job`, `education`, `contact`, `poutcome` | Нужно исследовать |


#### Оптимизация памяти

In [60]:
# Анализ типов данных
df.dtypes.value_counts()

object    10
int64      7
Name: count, dtype: int64

In [61]:
# Замер памяти до оптимизации
memory_before = df.memory_usage(deep=True).sum() / 1024**2

# Числовые типы
# выбрать атоматически
int_cols = df.select_dtypes(include=['int64']).columns

for col in int_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], downcast='integer')

# Категориальные типы (уникальность < 20%)
cat_candidates = df.select_dtypes(include=['object']).columns
for col in cat_candidates:
    if col in df.columns:
        unique_ratio = df[col].nunique() / len(df)
        if unique_ratio < 0.2:
            df[col] = df[col].astype('category')
            print(f" {col} → category (уникальность: {unique_ratio:.1%})")

# Замер памяти после оптимизации
memory_after = df.memory_usage(deep=True).sum() / 1024**2

# Вывод
print(f"\n💾 Память до: {memory_before:.2f} MB")
print(f"\n💾 Память после: {memory_after:.2f} MB")
print(f"\n📉 Экономия: {(1 - memory_after/memory_before)*100:.1f}%")

 job → category (уникальность: 0.0%)
 marital → category (уникальность: 0.0%)
 education → category (уникальность: 0.0%)
 default → category (уникальность: 0.0%)
 housing → category (уникальность: 0.0%)
 loan → category (уникальность: 0.0%)
 contact → category (уникальность: 0.0%)
 month → category (уникальность: 0.0%)
 poutcome → category (уникальность: 0.0%)
 y → category (уникальность: 0.0%)

💾 Память до: 27.61 MB

💾 Память после: 1.00 MB

📉 Экономия: 96.4%


In [62]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45211 entries, 0 to 45210
Data columns (total 17 columns):
 #   Column       Non-Null Count  Dtype   
---  ------       --------------  -----   
 0   age          45211 non-null  int8    
 1   job          44923 non-null  category
 2   marital      45211 non-null  category
 3   education    43354 non-null  category
 4   default      45211 non-null  category
 5   balance      45211 non-null  int32   
 6   housing      45211 non-null  category
 7   loan         45211 non-null  category
 8   contact      32191 non-null  category
 9   day_of_week  45211 non-null  int8    
 10  month        45211 non-null  category
 11  duration     45211 non-null  int16   
 12  campaign     45211 non-null  int8    
 13  pdays        45211 non-null  int16   
 14  previous     45211 non-null  int16   
 15  poutcome     8252 non-null   category
 16  y            45211 non-null  category
dtypes: category(10), int16(3), int32(1), int8(3)
memory usage: 1017.4 KB


**Выявлена ошибка именования** 
day_of_week - явно обозначает день месяца


In [63]:
df = df.rename(columns={'day_of_week': 'day_of_month'})

### Статистическая сводка

In [64]:
# Числовые переменные
df.describe().T

,count,mean,std,min,25%,50%,75%,max
age,45211.0,40.936210,10.618762,18.0,33.0,39.0,48.0,95.0
balance,45211.0,1362.272058,3044.765829,-8019.0,72.0,448.0,1428.0,102127.0
day_of_month,45211.0,15.806419,8.322476,1.0,8.0,16.0,21.0,31.0
duration,45211.0,258.163080,257.527812,0.0,103.0,180.0,319.0,4918.0
campaign,45211.0,2.763841,3.098021,1.0,1.0,2.0,3.0,63.0
pdays,45211.0,40.197828,100.128746,-1.0,-1.0,-1.0,-1.0,871.0
previous,45211.0,0.580323,2.303441,0.0,0.0,0.0,0.0,275.0


In [65]:
# Категориальные переменные
df.describe(include=['category']).T

,count,unique,top,freq
job,44923,11,blue-collar,9732
marital,45211,3,married,27214
education,43354,3,secondary,23202
default,45211,2,no,44396
housing,45211,2,yes,25130
loan,45211,2,no,37967
contact,32191,2,cellular,29285
month,45211,12,may,13766
poutcome,8252,3,failure,4901
y,45211,2,no,39922


## Качество данных